# Causal Softmax - Fused Single-Pass Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep03-softmax-causal.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series - Episode 3: Just Fuse It - RMSNorm and Softmax. Full write-up: [ep03-rmsnorm-softmax-fused.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep03-rmsnorm-softmax-fused.md).

Softmax runs 32 times per Bielik forward pass and, like RMSNorm, is memory-bound rather than compute-bound. This notebook runs the single-pass fused causal-softmax Triton kernel (`kernels/attention/softmax_causal_simple.py`).

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU - no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
import torch.nn.functional as F
from kernels.attention.softmax_causal_simple import softmax_causal_simple

torch.manual_seed(0)
batch, num_heads, seq_len = 2, 4, 128
scores = torch.randn(batch, num_heads, seq_len, seq_len, device="cuda", dtype=torch.float32)

def softmax_causal_ref(x):
    seq_len = x.shape[-1]
    mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1)
    x = x.masked_fill(mask, float("-inf"))
    return F.softmax(x, dim=-1)

out_triton = softmax_causal_simple(scores)
out_ref = softmax_causal_ref(scores)

max_diff = (out_triton - out_ref).abs().max().item()
print(f"max abs diff vs PyTorch reference: {max_diff:.6f}")
assert torch.allclose(out_triton, out_ref, atol=1e-5, rtol=1e-5)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-softmax` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

In [ ]:
!PYTHONPATH=. python benchmarks/attention/benchmark_softmax_causal.py --save-plots --plot-dir=docs/plots/attention/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/attention/"
for fname in [
    "softmax-causal-bandwidth-vs-heads.png",
    "softmax-causal-bandwidth-vs-seq-len.png",
    "softmax_causal-summary-bielik-config-gb_s.png",
    "softmax_causal-summary-bielik-config-tflops.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep03-rmsnorm-softmax-fused.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)